# GenHMM1d — R / Python parity check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamadouyamar/GenHMM1d/blob/master/parity.ipynb)

This notebook checks that the **Python GenHMM1d** (this repository) and the
**R GenHMM1d** ([CRAN](https://cran.r-project.org/package=GenHMM1d)) produce
the same estimates on the same data:

1. observations are **simulated with the R package** (`SimHMMGen`),
2. the **same series** is estimated with the R `EstHMMGen` and with the Python
   `EstHMMGen`,
3. the two estimates are compared parameter by parameter, against each other
   and against the truth.

Both implementations use the same deterministic block initialization and the
same EM, so with matched settings (`eps = 1e-4`, `max_iter = 10000`, minimum
100 EM iterations) the estimates should agree to numerical tolerance; the
only remaining differences come from the Nelder--Mead internals of
`stats::optim` vs `scipy.optimize.minimize`. Models covered: Gaussian,
Poisson, zero-inflated Gaussian, zero-inflated Poisson (the CRAN package has
no autoregressive models, so the AR classes of the Python package are out of
scope here).

**Run it on Google Colab** (badge above): the R side runs through `rpy2`.
Installing the R package and its dependencies takes a few minutes.


In [ ]:
# ---- Python package (this repo) --------------------------------------------
try:
    import genhmm1d
except ImportError:                       # e.g. on Google Colab
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/mamadouyamar/GenHMM1d.git"], check=True)

# ---- R package (CRAN), through rpy2 ----------------------------------------
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
numpy2ri.activate()

ro.r('options(repos = "https://cloud.r-project.org")')
ro.r('if (!requireNamespace("GenHMM1d", quietly = TRUE)) install.packages("GenHMM1d")')
ro.r('suppressMessages(library(GenHMM1d))')
print("R GenHMM1d version:", ro.r('as.character(packageVersion("GenHMM1d"))')[0])

import numpy as np
from genhmm1d.hmm import HMM
hmm = HMM()

N        = 2000      # sample size
MAX_ITER = 10000
EPS      = 1e-4      # R default stopping criterion; passed to both sides
NINIT    = 100       # R hardcodes 100 minimum EM iterations; matched in Python
TOL      = 1e-2      # parity tolerance on parameters (optimizer internals differ)


In [ ]:
# ---- parity helper ----------------------------------------------------------
def order_regimes(theta, Q, ZI):
    """Sort non-zero regimes by their first parameter (mean / lambda)."""
    theta, Q = np.asarray(theta, float), np.asarray(Q, float)
    k0 = ZI                                   # zero regime (if any) stays first
    order = np.concatenate([np.arange(k0),
                            k0 + np.argsort(theta[k0:, 0])]).astype(int)
    return theta[order], Q[np.ix_(order, order)]

def parity(name, family_r, family_py, theta_true, Q_true, ZI=0, seed=12345):
    reg = len(theta_true)
    theta_true = np.asarray(theta_true, float)
    Q_true = np.asarray(Q_true, float)

    # 1) simulate with the R package
    ro.globalenv["theta_r"] = theta_true
    ro.globalenv["Q_r"] = Q_true
    ro.r(f'set.seed({seed})')
    ro.r(f'sim <- GenHMM1d::SimHMMGen(theta_r, Q = Q_r, ZI = {ZI}, '
         f'family = "{family_r}", n = {N})')
    y = np.asarray(ro.r('as.numeric(sim$SimData)'))

    # 2) estimate with the R package
    ro.r(f'est <- GenHMM1d::EstHMMGen(sim$SimData, ZI = {ZI}, reg = {reg}, '
         f'family = "{family_r}", max_iter = {MAX_ITER}, eps = {EPS})')
    th_R = np.asarray(ro.r('est$theta'), float).reshape(reg, -1)
    Q_R  = np.asarray(ro.r('est$Q'), float).reshape(reg, reg)

    # 3) estimate the SAME series with the Python package
    out = hmm.EstHMMGen(y.reshape(-1, 1), reg, family_py,
                        max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=ZI)
    th_P = np.asarray(out["theta"], float).reshape(reg, -1)
    Q_P  = np.asarray(out["Q"], float)

    # 4) align regime labels and compare
    th_R, Q_R = order_regimes(th_R, Q_R, ZI)
    th_P, Q_P = order_regimes(th_P, Q_P, ZI)
    d_theta = np.max(np.abs(th_R - th_P))
    d_Q     = np.max(np.abs(Q_R - Q_P))

    print(f"== {name}  (n={N}, same R-simulated series) ==")
    print("theta_true:\n", np.round(theta_true, 4))
    print("theta_R   :\n", np.round(th_R, 4))
    print("theta_Py  :\n", np.round(th_P, 4))
    print("Q_true:\n", np.round(Q_true, 4))
    print("Q_R   :\n", np.round(Q_R, 4))
    print("Q_Py  :\n", np.round(Q_P, 4))
    verdict = "PASS" if max(d_theta, d_Q) < TOL else "FAIL -> investigate"
    print(f"max |R - Py|:  theta {d_theta:.2e}   Q {d_Q:.2e}   [{verdict}]\n")
    return d_theta, d_Q

Q2 = np.array([[0.94, 0.06],
               [0.03, 0.97]])
results = {}


## Gaussian, two regimes

In [ ]:
theta = np.array([[0.0,   1.0],
                  [1.349, 1.0]])            # [mu, sd] per regime (50% overlap)
results["gaussian"] = parity("Gaussian 2reg", "gaussian", "norm", theta, Q2)


## Poisson, two regimes

In [ ]:
theta = np.array([[2.0], [9.0]])            # [lambda] per regime
results["poisson"] = parity("Poisson 2reg", "poisson", "poisson", theta, Q2)


## Zero-inflated Gaussian, two regimes

In [ ]:
theta = np.array([[0.0, 0.0],
                  [3.0, 1.0]])              # row 1 = point mass at 0
results["zi-gaussian"] = parity("ZI-Gaussian 2reg", "gaussian", "norm",
                                theta, Q2, ZI=1)


## Zero-inflated Poisson, two regimes

In [ ]:
theta = np.array([[0.0], [9.0]])            # row 1 = point mass at 0
results["zi-poisson"] = parity("ZI-Poisson 2reg", "poisson", "poisson",
                               theta, Q2, ZI=1)


## Parity summary

In [ ]:
# ---- summary ----------------------------------------------------------------
print(f"{'model':<14}{'max|dtheta|':>14}{'max|dQ|':>12}   verdict")
for k, (dt, dq) in results.items():
    v = "PASS" if max(dt, dq) < TOL else "FAIL"
    print(f"{k:<14}{dt:>14.2e}{dq:>12.2e}   {v}")


### Reading the result

- **PASS everywhere**: on identical data, the Python package reproduces the
  CRAN R package to numerical tolerance for the iid and zero-inflated models
  --- the two implementations are the same estimator, and the Python package
  additionally provides the autoregressive models M1--M4 and the
  regime-switching copulas.
- **A FAIL is informative, not cosmetic**: with identical data and
  deterministic initialization on both sides, a discrepancy beyond optimizer
  tolerance points to a genuine difference between the code bases (for
  instance a correction present in one and not the other) and identifies
  exactly which family and parameter to inspect.
